# 2.0 — Procesamiento: temperatura SMN/CONAGUA

**Capa 2 del protocolo (clima).** Convierte los resúmenes mensuales por
estación de `01_ingesta/1.0-conagua-temperatura.ipynb` (TMed, TMax, TMin)
en una sola serie mensual para Hermosillo, la que usa el panel integrado
(`03_eda/3.0-integracion-espacial.ipynb`).

## Entradas

| Ruta | Contenido |
|------|-----------|
| `data/raw/temperatura_AAAA/*.csv` | Temperatura media mensual por estación (`Tmed`) |
| `data/raw/temperatura_max_AAAA/*.csv` | Temperatura máxima mensual por estación (`Tmax`) |
| `data/raw/temperatura_min_AAAA/*.csv` | Temperatura mínima mensual por estación (`Tmin`) |

## Salidas

- `data/interim/clima/temperatura_smn_hermosillo_mensual.csv` (opcional,
  para revisión): `fecha_mes`, `temp_media_c`, `temp_max_media_c`,
  `temp_min_media_c`.
- Decisiones de limpieza registradas al final del notebook.

## Funciones del paquete

`panel.load_smn_temperature()`, `load_smn_temperature_max()`,
`load_smn_temperature_min()` y `panel.climate_hermosillo()` (filtra las
estaciones de `config.CLIMATE_STATION_IDS`, promedia y marca los meses
corruptos).

## Checklist

- [ ] Cobertura: meses con dato por estación de Hermosillo.
- [ ] Valores centinela del SMN (`-99.9`, `-999`) y rangos físicos (`Tmin ≤ Tmed ≤ Tmax`).
- [ ] Meses corruptos en la fuente (mismo valor en TMed/TMax/TMin) → NaN.
- [ ] Promedio entre estaciones y número de estaciones que aporta cada mes.
- [ ] Faltantes en la serie mensual final (se tratan en `03_eda/3.1-datos-faltantes.ipynb`).

## Configuración e imports

In [ ]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sonora_health_climate_eda.config import CLIMATE_STATION_IDS, INTERIM_DATA_DIR
from sonora_health_climate_eda.panel import (
    climate_hermosillo,
    load_smn_temperature,
    load_smn_temperature_max,
    load_smn_temperature_min,
)

OUT_DIR = INTERIM_DATA_DIR / "clima"
print("Estaciones de Hermosillo:", CLIMATE_STATION_IDS)

## Carga por estación

In [ ]:
tmed = load_smn_temperature()
tmax = load_smn_temperature_max()
tmin = load_smn_temperature_min()

for nombre, df in {"TMed": tmed, "TMax": tmax, "TMin": tmin}.items():
    print(
        f"{nombre}: {len(df):,} filas estación-mes, {df['clave'].nunique()} estaciones, "
        f"{df['fecha_mes'].min()} a {df['fecha_mes'].max()}"
    )

## Cobertura de las estaciones de Hermosillo

Un mes sin ninguna estación con dato queda como NaN después del promedio.

In [ ]:
herm = tmed[tmed["clave"].isin(CLIMATE_STATION_IDS)]
cobertura = (
    herm.dropna(subset=["temp_media_c"])
    .groupby(["fecha_mes", "clave"])
    .size()
    .unstack("clave")
)
print("Meses con dato por estación (TMed):")
print(cobertura.notna().sum())

n_estaciones = cobertura.notna().sum(axis=1)
print("\nMeses según número de estaciones con dato:")
print(n_estaciones.value_counts().sort_index())

## Valores centinela y rangos físicos

In [ ]:
SERIES = [("TMed", tmed, "temp_media_c"), ("TMax", tmax, "temp_max_media_c"), ("TMin", tmin, "temp_min_media_c")]
for nombre, df, col in SERIES:
    n_centinela = df[col].isin([-99.9, -999.0, -99.0]).sum()
    print(f"{nombre}: centinelas={n_centinela}, min={df[col].min():.1f}, max={df[col].max():.1f}, NaN={df[col].isna().sum()}")

# Une las tres series por estación-mes para revisar coherencia física
llave = ["clave", "fecha_mes"]
est = (
    tmed[llave + ["estacion", "temp_media_c"]]
    .merge(tmax[llave + ["temp_max_media_c"]], on=llave, how="inner")
    .merge(tmin[llave + ["temp_min_media_c"]], on=llave, how="inner")
)
incoherentes = est[
    (est["temp_min_media_c"] > est["temp_media_c"]) | (est["temp_media_c"] > est["temp_max_media_c"])
]
print(f"\nEstación-mes con Tmin > Tmed o Tmed > Tmax: {len(incoherentes)}")
incoherentes[incoherentes["clave"].isin(CLIMATE_STATION_IDS)].head(20)

## Meses corruptos en la fuente

El SMN publicó archivos degenerados para dic-2015, dic-2016 y dic-2017: cada
estación trae el mismo valor en TMed, TMax y TMin (físicamente imposible) y
~8–10 °C por encima del diciembre típico. `climate_hermosillo()` los marca
como NaN. Aquí se buscan otros meses con el mismo patrón.

In [ ]:
mismo_valor = est[
    np.isclose(est["temp_media_c"], est["temp_max_media_c"])
    & np.isclose(est["temp_media_c"], est["temp_min_media_c"])
]
print("Meses con más estaciones con TMed = TMax = TMin:")
mismo_valor.groupby("fecha_mes")["clave"].nunique().sort_values(ascending=False).head(10)

# TODO: si aparecen meses nuevos con este patrón, agregarlos a `bad_months`
# en panel.climate_hermosillo() y registrarlo en la tabla de decisiones.

## Serie mensual de Hermosillo

In [ ]:
temp_mensual = climate_hermosillo()
print(temp_mensual.isna().sum())
temp_mensual.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for col in ["temp_min_media_c", "temp_media_c", "temp_max_media_c"]:
    ax.plot(temp_mensual["fecha_mes"], temp_mensual[col], label=col)
ax.set_ylabel("°C")
ax.set_title("Temperatura mensual SMN — Hermosillo (promedio de estaciones)")
ax.legend()
plt.tight_layout()
plt.show()

# TODO: agregar `n_estaciones` a la salida de climate_hermosillo() si la
# cobertura varía mucho entre meses (un promedio de 1 estación no es igual
# de confiable que uno de 3).

## Guardar salida

In [ ]:
# TODO: guardar una vez validada la serie
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# temp_mensual.to_csv(OUT_DIR / "temperatura_smn_hermosillo_mensual.csv", index=False)

## Decisiones de procesamiento

Registrar cada decisión que cambie los datos y su justificación. Si la
decisión cambia el panel, implementarla en `sonora_health_climate_eda/panel.py`
(es lo que ejecuta `03_eda/3.0-integracion-espacial.ipynb`) y llevarla al
reporte final.

| Variable / periodo | Problema | Decisión | Justificación |
|--------------------|----------|----------|---------------|
| dic-2015, dic-2016, dic-2017 (TMed/TMax/TMin) | Archivo SMN degenerado | NaN | Valor idéntico en las tres series y ~8–10 °C sobre el diciembre típico |
| _TODO_ | | | |